# 2.2 Exercises

Based on [2.2-ml-model-quality.ipynb](2.2-ml-model-quality.ipynb). Try each one before looking anything up.

Useful imports for single metrics instead of presets:

```python
from evidently.metrics import Accuracy, Precision, Recall, F1Score, RocAuc, MeanError, MAE, RMSE, MAPE
```

A result's values are in `result.dict()["metrics"]` – a list of `{"metric_name": ..., "value": ...}`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets, ensemble, linear_model
from evidently import BinaryClassification, DataDefinition, Dataset, Regression, Report, Recsys
from evidently.presets import ClassificationPreset, RegressionPreset
from evidently.metrics import (
    Accuracy, Precision, Recall, F1Score, RocAuc,
    MeanError, MAE, RMSE, MAPE,
    PrecisionTopK, RecallTopK, NDCG, MAP, MRR, HitRate,
)

## Part 1 – Classification

Set up the same breast cancer data and random forest as in the lesson notebook (`reference`, `current`, `prediction` column).

In [ ]:
# Setup

### 1.1 Malignant as the positive class

In this dataset `0` = malignant. Missing a malignant case is the costly error.

- Build a `DataDefinition` with `pos_label=0`.
- Keep `prediction` as the probability of class **1** – Evidently always reads a single probability column as P(class 1). `pos_label` only changes which class the metrics are reported for.
- Run `Precision`, `Recall`, `F1Score` with `probas_threshold=0.5` for both `pos_label=1` and `pos_label=0`.

**Question:** which class does the model find more reliably?

### 1.2 Pick a threshold

Loop over thresholds `[0.1, 0.3, 0.5, 0.7, 0.9]` and collect precision and recall (for `pos_label=1`) into a DataFrame with columns `threshold, precision, recall`.

**Question:** which threshold would you pick if a missed benign case is cheap but a false alarm is expensive?

### 1.3 Simulate a broken feature

Copy `current` and break it: multiply `worst radius` and `worst area` by 3 (e.g. a unit change upstream). Recompute the predictions on the broken data and run `ClassificationPreset` against `reference`.

**Question:** which metrics dropped? Would you have noticed this without labels?

### 1.4 Quality by segment

Split `current` into two segments: `mean radius` above and below the median. Run `Accuracy`, `Precision`, `Recall` on each segment separately and put the results side by side in one DataFrame.

**Question:** does the overall score hide a weaker segment?

## Part 2 – Regression

### 2.1 Train a real model

Replace the fake "target + noise" predictions with a real model:
- Load California housing, rename `MedHouseVal` to `target`.
- Split into `housing_reference` / `housing_current` (e.g. 5000 rows each, different `random_state`).
- Train `linear_model.LinearRegression` on reference features, add a `prediction` column to both.
- Run `RegressionPreset`.

### 2.2 Over- vs underestimation

Make a biased copy of `housing_current` where `prediction = prediction * 1.2`. Run `MeanError`, `MAE`, `RMSE`, `MAPE` on the original and biased versions.

**Question:** which metric tells you the model *over*-predicts, and which only says the error got bigger?

### 2.3 Drift in production

Create a current dataset of only expensive districts: `target > housing["target"].quantile(0.8)`. Run `RegressionPreset` against the full reference.

**Question:** is the model worse on this segment? What in the report shows it?

## Part 3 – Ranking

Copy `make_recs` from the lesson notebook.

### 3.1 Effect of k

Using `quality=0.6`, run `PrecisionTopK`, `RecallTopK`, `NDCG` for `k = 1, 3, 5, 10`.

**Question:** why does recall go up with k while precision goes down?

### 3.2 Detect a worse model

For `quality` in `[0.9, 0.7, 0.5, 0.3, 0.1]` compute `NDCG(k=5)` and `HitRate(k=5)`, collect into a DataFrame and plot NDCG vs quality.

**Question:** at what quality would you raise an alert if the reference model is `quality=0.8`?

### 3.3 Graded relevance

Change `make_recs` so `target` is a rating from 0 to 3 instead of 0/1 (e.g. `int(rel * 4)`). Run `NDCG(k=5)` and `PrecisionTopK(k=5, min_rel_score=2)`.

**Question:** what does `min_rel_score` do, and why does NDCG work without it?

## Part 4 – Using results

### 4.1 Metrics table

Run a classification report with `Accuracy`, `Precision`, `Recall`, `F1Score`, `RocAuc` on both `reference` and `current` (as two separate runs with no reference data). Build a DataFrame with columns `metric, reference, current, change`.

### 4.2 Simple alert

Write a function `check_quality(result, min_recall=0.9)` that reads recall from `result.dict()` and prints `ALERT` if it is below the limit. Test it on the broken data from 1.3.